## Libraries

In [104]:
import random
import numpy as np
import torch
from pathlib import Path
import kagglehub
from dataclasses import dataclass, replace, asdict
import json
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.utils as vutils
from PIL import Image
import matplotlib.pyplot as plt
import time
import torch.nn as nn
import torch.optim as optim

## Config Cell

In [105]:
@dataclass(frozen= True)
class Config:
    # Reproducibility
    SEED: int = 42

    # Dataset
    DATASET: str = "jessicali9530/celeba-dataset"
    EXPECTED_IMAGES: int = 202_599
    DATA_DIR: Path = (Path.cwd().parent / "Data" / "celeba").resolve()

    # Run / output paths
    RUN_NAME: str = "baseline"
    ARTIFACT_ROOT: Path = (Path.cwd() / "artifacts" / "GAN").resolve()

    # Data Pipeline
    IMAGE_SIZE: int = 64
    CHANNELS: int = 3
    BATCH_SIZE: int = 128
    NUM_WORKERS: int = 4
    CROP_SIZE: int = 148

    # Model
    LATENT_DIM: int = 100
    G_FEATURES: int = 64
    D_FEATURES: int = 64

    # Optimizer / init 
    LR: float = 2e-4
    BETA1: float = 0.5
    BETA2: float = 0.999
    INIT_STD: float = 0.02

    # Training
    EPOCHS: int = 25
    LOG_EVERY: int = 100

    # --- Monitoring ---
    N_FIXED_SAMPLES: int = 64
    
    def __post_init__(self):
        assert self.IMAGE_SIZE == 64

    @property
    def RUN_DIR(self) -> Path:
        return self.ARTIFACT_ROOT / self.RUN_NAME

    @property
    def CKPT_DIR(self) -> Path:
       return self.RUN_DIR / "checkpoints" 

    @property
    def SAMPLE_DIR(self) -> Path:
        return self.RUN_DIR / "samples"

    @property
    def PLOT_DIR(self) -> Path:
        return self.RUN_DIR / "plots"

    def make_dirs(self):
        for d in (self.RUN_DIR, self.CKPT_DIR,self.SAMPLE_DIR, self.PLOT_DIR):
            d.mkdir(parents = True, exist_ok = True)

    def save(self):
        path = self.RUN_DIR / "config.json"
        path.write_text(json.dumps(asdict(self), indent = 2, default = str))
        return path

cfg = Config()
cfg.make_dirs()
cfg.save()
print(f"Run: {cfg.RUN_NAME} -> {cfg.RUN_DIR}")


Run: baseline -> /home/harsh/Work/Notebooks/Projects/Notebooks/artifacts/GAN/baseline


In [106]:
# Checks For GPU
seed = cfg.SEED
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)   # safe to call even without a GPU

# Falls back to CPU so other people can still run the notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA (built with):", torch.version.cuda)
    print(f"VRAM free/total: {free / 1e9:.1f} / {total / 1e9:.1f} GB")

PyTorch: 2.14.1+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 3060
CUDA (built with): 13.0
VRAM free/total: 11.0 / 12.9 GB


## Data 

In [107]:
# Relative to the notebook (Projects/Notebooks), so it works on any machine
data = cfg.DATA_DIR
image_count = cfg.EXPECTED_IMAGES

data.mkdir(parents=True, exist_ok=True)

def count_jpgs(root):
    return sum(1 for _ in root.rglob("*.jpg"))

existing = count_jpgs(data)

if existing >= image_count:
    print(f"Dataset already present ({existing:,} images). Skipping download.")
else:
    print(f"Found {existing:,} images, expected {image_count:,}. Downloading...")
    kagglehub.dataset_download(
        cfg.DATASET,
        output_dir=str(data),
        force_download=existing > 0,   # re-fetch only if a partial copy exists
    )

# Locate the folder that actually holds the JPEGs (the zip may nest it twice)
first = next(data.rglob("*.jpg"), None)
if first is None:
    raise FileNotFoundError(f"No JPEGs found under {data}. Download failed.")

IMG_DIR = first.parent
n_images = sum(1 for _ in IMG_DIR.glob("*.jpg"))
if n_images < image_count:
    raise RuntimeError(f"Only {n_images:,} images in {IMG_DIR}. Delete {data} and rerun.")

print(f"Ready: {n_images:,} images in {IMG_DIR}")


Dataset already present (202,599 images). Skipping download.
Ready: 202,599 images in /home/harsh/Work/Notebooks/Projects/Data/celeba/img_align_celeba/img_align_celeba


## Pipeline

In [ ]:
class CelebADataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.paths = sorted(Path(img_dir).glob("*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB") 
        return self.transform(img) if self.transform else img

transform = transforms.Compose([
    transforms.CenterCrop(cfg.CROP_SIZE),
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * cfg.CHANNELS, [0.5] * cfg.CHANNELS),
])

dataset = CelebADataset(IMG_DIR, transform)

loader = DataLoader(
    dataset,
    batch_size= cfg.BATCH_SIZE,
    shuffle= True,
    drop_last= True,
    num_workers= cfg.NUM_WORKERS,
    pin_memory= True,
    persistent_workers= cfg.NUM_WORKERS > 0,
)

print(f"Images: {len(dataset):,} | Batches/epoch: {len(loader):,}")

## Model Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        nz, f, nc = cfg.LATENT_DIM, cfg.G_FEATURES, cfg.CHANNELS
        self.nz = nz
        self.net = nn.Sequential(
            # [B, nz, 1, 1] -> [B, 8f, 4, 4]
            nn.ConvTranspose2d(nz, f * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(f * 8), nn.ReLU(True),
            # -> [B, 4f, 8, 8]
            nn.ConvTranspose2d(f * 8, f * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(f * 4), nn.ReLU(True),
            # -> [B, 2f, 16, 16]
            nn.ConvTranspose2d(f * 4, f * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(f * 2), nn.ReLU(True),
            # -> [B, f, 32, 32]
            nn.ConvTranspose2d(f * 2, f, 4, 2, 1, bias=False),
            nn.BatchNorm2d(f), nn.ReLU(True),
            # -> [B, nc, 64, 64], values in [-1, 1]
            nn.ConvTranspose2d(f, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), self.nz, 1, 1))
        

## Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        nc, f = cfg.CHANNELS, cfg.D_FEATURES
        self.net = nn.Sequential(
            nn.Conv2d(nc, f, 4, 2, 1, bias= False),
            nn.LeakyReLU(0.2, inplace= True),

            nn.Conv2d(f, f * 2, 4, 2, 1, bias= False),
            nn.BatchNorm2d(f*2), nn.LeakyReLU(0.2, inplace= True),

            nn.Conv2d(f*2, f*4, 4, 2, 1, bias= False),
            nn.BatchNorm2d(f*4), nn.LeakyReLU(0.2, inplace= True),

            nn.Conv2d(f*4, f*8, 4, 2, 1, bias= False),
            nn.BatchNorm2d(f*8), nn.LeakyReLU(0.2, inplace= True),

            nn.Conv2d(f*8, 1, 4, 1, 0, bias= False)
        )

    def forward(self, x):
        out = self.net(x)
        assert out.shape[1:] == (1, 1, 1), f"Unexpected D output: {tuple(out.shape)}"
        return out.view(-1)

## Optimizer Cell

In [ ]:
def weights_init(m):
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight, 0.0, cfg.INIT_STD)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight, 1.0, cfg.INIT_STD)
        nn.init.zeros_(m.bias)

netG = Generator(cfg).to(device)
netD = Discriminator(cfg).to(device)
netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCEWithLogitsLoss()
optG = optim.Adam(netG.parameters(), lr=cfg.LR, betas=(cfg.BETA1, cfg.BETA2))
optD = optim.Adam(netD.parameters(), lr=cfg.LR, betas=(cfg.BETA1, cfg.BETA2))

## Training

In [ ]:
fixed_gen = torch.Generator().manual_seed(cfg.SEED)
fixed_noise = torch.randn(cfg.N_FIXED_SAMPLES, cfg.LATENT_DIM, generator=fixed_gen).to(device)

@torch.no_grad()
def save_samples(epoch, cfg):
    fake = netG(fixed_noise).cpu()
    path = cfg.SAMPLE_DIR / f"epoch_{epoch:03d}.png"
    vutils.save_image(fake, path, nrow=8, normalize=True, value_range=(-1, 1))
    return path

def plot_history(history, cfg):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history["step"], history["d_loss"], label="D loss")
    ax[0].plot(history["step"], history["g_loss"], label="G loss")
    ax[0].set_xlabel("step"); ax[0].legend()
    ax[1].plot(history["step"], history["d_real"], label="D(real)")
    ax[1].plot(history["step"], history["d_fake"], label="D(fake)")
    ax[1].set_xlabel("step"); ax[1].legend()
    fig.tight_layout()
    path = cfg.PLOT_DIR / "history.png"
    fig.savefig(path)
    plt.show()
    return path

In [ ]:
def train(cfg, epochs=None, max_steps=None):
    history = {"step": [], "d_loss": [], "g_loss": [], "d_real": [], "d_fake": []}
    epochs = epochs or cfg.EPOCHS
    step = 0
    netG.train(); netD.train()

    for epoch in range(epochs):
        t0 = time.time()
        for real in loader:
            real = real.to(device, non_blocking=True)
            b = real.size(0)
            ones = torch.ones(b, device=device)
            zeros = torch.zeros(b, device=device)

            # ---- Update D: real -> 1, fake -> 0 ----
            optD.zero_grad(set_to_none=True)
            out_real = netD(real)
            loss_real = criterion(out_real, ones)

            fake = netG(torch.randn(b, cfg.LATENT_DIM, device=device))
            out_fake = netD(fake.detach())
            loss_fake = criterion(out_fake, zeros)

            d_loss = loss_real + loss_fake
            d_loss.backward()
            optD.step()

            # ---- Update G: make D say "real" (label 1) ----
            optG.zero_grad(set_to_none=True)
            g_loss = criterion(netD(fake), ones)
            g_loss.backward()
            optG.step()

            if step % cfg.LOG_EVERY == 0:
                history["step"].append(step)
                history["d_loss"].append(d_loss.item())
                history["g_loss"].append(g_loss.item())
                history["d_real"].append(torch.sigmoid(out_real).mean().item())
                history["d_fake"].append(torch.sigmoid(out_fake).mean().item())
                print(f"ep {epoch+1}/{epochs} step {step} | "
                      f"D {history['d_loss'][-1]:.3f} G {history['g_loss'][-1]:.3f} | "
                      f"D(real) {history['d_real'][-1]:.2f} D(fake) {history['d_fake'][-1]:.2f}")

            step += 1
            if max_steps and step >= max_steps:
                return history

        save_samples(epoch + 1, cfg)
        print(f"Epoch {epoch+1} done in {time.time() - t0:.0f}s")
    return history

In [ ]:
save_samples(0, cfg)                  
history = train(cfg, epochs=1)
plot_history(history, cfg)